# ⚡ Flexibility Valorization Analysis — Dutch Grid Capacity Data

**Goal:** Identify postcodes and supply areas with the highest commercial potential for energy flexibility programs (demand-response, load-shifting, feed-in valorization).

---

## How to read this notebook

Each section follows the same structure:
1. 🔵 **What we do** — the code
2. 🟡 **Why we do it** — the analytical rationale
3. 🟢 **What to look for** — how to interpret the output

The sections build on each other. By the end you will have a ranked list of postcodes with a composite *Valorization Score* ready for commercial targeting.


---
## 0 · Environment Setup

Import the only two libraries we need and configure display options.


In [1]:
import pandas as pd
import numpy as np

# Show up to 120 characters per row so wide DataFrames don't truncate
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

print("✓ Libraries loaded")


✓ Libraries loaded


---
## 1 · Load Raw Data

We load five CSV files.  All files use `;` as separator and `,` as decimal — a common Dutch/European convention.

| Variable | File | Description |
|---|---|---|
| `pc6` | `congestie_pc6.csv` | One row per 6-digit postcode with congestion color codes |
| `vg` | `voedingsgebieden.csv` | Supply-area capacity, queue, and request data |
| `tg` | `tennetgebieden.csv` | TenneT high-voltage area data |
| `tc` | `tennetcongestie.csv` | TenneT substation congestion codes |
| `proj` | `projecten.csv` | Planned grid upgrade projects |


In [2]:
pc6  = pd.read_csv("raw_data/congestie_pc6.csv",      sep=";", decimal=",")
vg   = pd.read_csv("raw_data/voedingsgebieden.csv",   sep=";", decimal=",")
tg   = pd.read_csv("raw_data/tennetgebieden.csv",     sep=";", decimal=",")
tc   = pd.read_csv("raw_data/tennetcongestie.csv",    sep=";", decimal=",")
proj = pd.read_csv("raw_data/projecten.csv",          sep=";", decimal=",")

print(f"pc6  : {pc6.shape[0]:>6,} rows × {pc6.shape[1]} cols")
print(f"vg   : {vg.shape[0]:>6,} rows × {vg.shape[1]} cols")
print(f"tg   : {tg.shape[0]:>6,} rows × {tg.shape[1]} cols")
print(f"tc   : {tc.shape[0]:>6,} rows × {tc.shape[1]} cols")
print(f"proj : {proj.shape[0]:>6,} rows × {proj.shape[1]} cols")


pc6  : 462,606 rows × 10 cols
vg   :    608 rows × 21 cols
tg   :    150 rows × 23 cols
tc   :    271 rows × 5 cols
proj :    945 rows × 17 cols


> **Quick check:** `pc6` should have several hundred thousand rows (one per postcode); `vg` typically has a few hundred supply areas. A large mismatch may indicate a loading issue.


In [3]:
# Preview each table
print("── pc6 ──────────────────────────────────")
display(pc6.head(3))
print("── vg ───────────────────────────────────")
display(vg.head(3))
print("── proj ─────────────────────────────────")
display(proj.head(3))


── pc6 ──────────────────────────────────


,postcode,afname,opwek,voedingsgebied_id,voedingsgebied_naam,tennet_id,RNB_postcode,Gemeentecode,Gemeentenaam,provincie
0,7471AA,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0000,Hof van Twente,Overijssel
1,7471AB,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0000,Hof van Twente,Overijssel
2,7471AC,1,1,GO,GO - Goor,Station Goor 110 kV,Coteq,1735.0000,Hof van Twente,Overijssel


── vg ───────────────────────────────────


,voedingsgebied_id,jaar,aanwezige_transportcapaciteit_invoeding,aanwezige_transportcapaciteit_afname,benodigde_transportcapaciteit_invoeding,benodigde_transportcapaciteit_afname,unieke_verzoeken_invoeding,unieke_verzoeken_afname,wachtrij_invoeding,wachtrij_afname,voorspelde_capaciteit_invoeding,voorspelde_capaciteit_afname,info,congestie_url,jaartal_opgelost_invoeding,jaartal_opgelost_afname,start,eind,kwartaal,RNB,provincie
0,118065277_23,2026,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stedin,Zuid-Holland
1,AMLM,2026,78.4900,83.5600,24.7148,79.3588,44.0000,42.0000,39.4180,10.5550,NaN,NaN,NaN,https://www.enexis.nl/zakelijk/netcapaciteit/c...,2025.0000,2029.0000,NaN,NaN,NaN,"Coteq, Enexis",Overijssel
2,AMLU,2026,62.4300,65.5500,26.5896,35.2893,35.0000,18.0000,36.0560,18.5140,NaN,NaN,NaN,https://www.enexis.nl/zakelijk/netcapaciteit/c...,2028.0000,2025.0000,NaN,NaN,NaN,"Coteq, Enexis",Overijssel


── proj ─────────────────────────────────


,Unnamed: 0,projectnaam,gebied_id,kwartaal,jaar,beschrijving,jaar_num,kwartaal_num,einddatum_string,netbeheerder,start,eind,tijdschema_beschrijving,project_fase,bron,project_url,project_type
0,NaN,Extra capaciteit op Californië,CALF,NaN,2029,NaN,NaN,NaN,2029,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN
1,NaN,Extra capaciteit op Nederweert,NEDW,NaN,2029,NaN,NaN,NaN,2029,Enexis,NaN,NaN,NaN,scheduled,Laatste inzichten,NaN,NaN
2,NaN,Extra capaciteit op Hengelo Bolderhoek,HGLB,NaN,2028,NaN,NaN,NaN,2028,Enexis,NaN,NaN,NaN,inProgress,Laatste inzichten,NaN,NaN


---
## 2 · Rename Columns to English

The source files use Dutch column names.  We rename everything to readable English equivalents before any further processing — this makes the rest of the notebook self-documenting.

> 💡 **Convention:** We keep the original Dutch names in the comments so you can always trace back to the source dictionary.


In [4]:
pc6 = pc6.rename(columns={
    "postcode":            "postcode",
    "afname":              "consumption_color",    # 0=transparent … 3=red
    "opwek":               "generation_color",
    "voedingsgebied_id":   "supply_area_id",
    "voedingsgebied_naam": "supply_area_name",
    "tennet_id":           "tennet_id",
    "RNB_postcode":        "regional_grid_operator",
})

vg = vg.rename(columns={
    "voedingsgebied_id":                        "supply_area_id",
    "aanwezige_transportcapaciteit_invoeding":  "available_feedin_capacity_mw",
    "aanwezige_transportcapaciteit_afname":     "available_consumption_capacity_mw",
    "benodigde_transportcapaciteit_invoeding":  "required_feedin_capacity_mw",
    "benodigde_transportcapaciteit_afname":     "required_consumption_capacity_mw",
    "unieke_verzoeken_invoeding":               "unique_feedin_requests",
    "unieke_verzoeken_afname":                  "unique_consumption_requests",
    "wachtrij_invoeding":                       "feedin_queue_mw",
    "wachtrij_afname":                          "consumption_queue_mw",
    "RNB":                                      "regional_grid_operator",
    "info":                                     "additional_info",
})

tg = tg.rename(columns={
    "aanwezige_transportcapaciteit_invoeding":  "available_feedin_capacity_mw",
    "aanwezige_transportcapaciteit_afname":     "available_consumption_capacity_mw",
    "benodigde_transportcapaciteit_invoeding":  "required_feedin_capacity_mw",
    "benodigde_transportcapaciteit_afname":     "required_consumption_capacity_mw",
    "unieke_verzoeken_invoeding":               "unique_feedin_requests",
    "unieke_verzoeken_afname":                  "unique_consumption_requests",
    "wachtrij_invoeding":                       "feedin_queue_mw",
    "wachtrij_afname":                          "consumption_queue_mw",
    "congestiegebied":                          "congestion_area",
    "info":                                     "additional_info",
})

tc = tc.rename(columns={
    "tennet_id":              "tennet_id",
    "congestiegebied_afname": "consumption_congestion_area",
    "congestiegebied_opwek":  "generation_congestion_area",
    "afname":                 "consumption_congestion",
    "opwek":                  "generation_congestion",
})

proj = proj.rename(columns={
    "projectnaam":  "project_name",
    "gebied_id":    "area_id",
    "kwartaal":     "completion_quarter",
    "jaar":         "completion_year",
    "beschrijving": "description",
    "einddatum":    "completion_date",
    "netbeheerder": "grid_operator",
})

print("✓ All columns renamed")
display(vg.columns.tolist())


✓ All columns renamed


['supply_area_id',
 'jaar',
 'available_feedin_capacity_mw',
 'available_consumption_capacity_mw',
 'required_feedin_capacity_mw',
 'required_consumption_capacity_mw',
 'unique_feedin_requests',
 'unique_consumption_requests',
 'feedin_queue_mw',
 'consumption_queue_mw',
 'voorspelde_capaciteit_invoeding',
 'voorspelde_capaciteit_afname',
 'additional_info',
 'congestie_url',
 'jaartal_opgelost_invoeding',
 'jaartal_opgelost_afname',
 'start',
 'eind',
 'kwartaal',
 'regional_grid_operator',
 'provincie']

> **Sanity check:** confirm the supply-area table (`vg`) now shows all eight capacity/queue columns in English before proceeding.


---
## 3 · Explore the Supply-Area Table (`vg`)

Before computing any metrics, we understand the distribution of the raw capacity numbers.

This step answers:
- Are there areas where `available = 0`? (Would cause division-by-zero later)
- What is the typical ratio of required vs available capacity?
- How large are the queues in practice?


In [5]:
print("── Descriptive statistics (vg) ─────────────────────────────────────────")
capacity_cols = [
    "available_feedin_capacity_mw", "available_consumption_capacity_mw",
    "required_feedin_capacity_mw",  "required_consumption_capacity_mw",
    "feedin_queue_mw",              "consumption_queue_mw",
]
display(vg[capacity_cols].describe().T)


── Descriptive statistics (vg) ─────────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
available_feedin_capacity_mw,340.0000,81.5974,59.9967,2.9000,41.7175,67.1050,99.7375,509.1200
available_consumption_capacity_mw,360.0000,72.1679,64.0936,0.0000,36.0000,52.5000,88.0000,518.0400
required_feedin_capacity_mw,338.0000,37.0256,45.2188,0.0000,10.0000,25.5824,49.0751,383.9470
required_consumption_capacity_mw,357.0000,48.8250,40.2938,1.0000,26.0000,38.0000,57.4274,310.8000
feedin_queue_mw,476.0000,11.3351,24.5019,0.0000,0.0000,2.0780,10.6260,321.5200
consumption_queue_mw,545.0000,13.8808,23.0436,0.0000,0.0000,5.2910,15.5684,159.7720


In [6]:
# How many supply areas have zero available capacity? (→ fully saturated)
zero_feedin      = (vg["available_feedin_capacity_mw"] == 0).sum()
zero_consumption = (vg["available_consumption_capacity_mw"] == 0).sum()
print(f"Areas with 0 available feed-in capacity    : {zero_feedin}")
print(f"Areas with 0 available consumption capacity: {zero_consumption}")
print()
print("These areas are maximally congested — top candidates for flexibility contracts.")


Areas with 0 available feed-in capacity    : 0
Areas with 0 available consumption capacity: 2

These areas are maximally congested — top candidates for flexibility contracts.


In [7]:
# Color-code distribution across postcodes
print("── Consumption color-code distribution (pc6) ───────────────────────────")
print(pc6["consumption_color"].value_counts().sort_index().rename({
    -1: "-1  No info", 0: " 0  Transparent", 1: " 1  Yellow",
     2: " 2  Orange",  3: " 3  Red"
}))
print()
print("── Generation color-code distribution (pc6) ────────────────────────────")
print(pc6["generation_color"].value_counts().sort_index().rename({
    -1: "-1  No info", 0: " 0  Transparent", 1: " 1  Yellow",
     2: " 2  Orange",  3: " 3  Red"
}))


── Consumption color-code distribution (pc6) ───────────────────────────
consumption_color
-1  No info            36
 0  Transparent     95986
 1  Yellow          85795
 2  Orange          36140
 3  Red            244649
Name: count, dtype: int64

── Generation color-code distribution (pc6) ────────────────────────────
generation_color
-1  No info            36
 0  Transparent    227729
 1  Yellow          93217
 2  Orange          83725
 3  Red             57899
Name: count, dtype: int64


> 🟢 **What to look for:** The proportion of orange + red postcodes tells you how widespread the congestion problem already is nationally. A high proportion (>30%) signals a sustained market for flexibility — this is not a local blip.


---
## 4 · Metric 1 — Capacity Utilisation Ratios

### Formula
$$\text{utilisation\_ratio} = \frac{\text{required\_capacity\_MW}}{\text{available\_capacity\_MW}}$$

### Why this is the primary signal

| Ratio value | Interpretation | Flexibility opportunity |
|---|---|---|
| < 1.0 | Headroom exists | Low — DSO can handle new connections |
| = 1.0 | Exactly at limit | Medium — one large connection tips it over |
| > 1.0 | **Over-subscribed** | **High — DSO has committed more than it can deliver** |
| >> 1.0 | Severely saturated | **Highest — multi-year contracts, premium rates** |

When `required > available`, the DSO has already signed connection agreements it cannot honour without either (a) building new infrastructure (slow, expensive) or (b) **paying for flexibility to shift/reduce load**. This is the core of the Dutch GOPACS/CODS market.

We add `EPSILON = 1e-9` to the denominator to safely handle areas reporting `available = 0`.


In [8]:
EPSILON = 1e-9  # prevents division-by-zero for fully saturated areas

# Feed-in utilisation: how oversubscribed is the export capacity?
vg["feedin_utilisation_ratio"] = (
    vg["required_feedin_capacity_mw"] /
    (vg["available_feedin_capacity_mw"] + EPSILON)
)

# Consumption utilisation: how oversubscribed is the import capacity?
vg["consumption_utilisation_ratio"] = (
    vg["required_consumption_capacity_mw"] /
    (vg["available_consumption_capacity_mw"] + EPSILON)
)

print("── Utilisation ratio distribution ───────────────────────────────────────")
display(vg[["feedin_utilisation_ratio", "consumption_utilisation_ratio"]].describe().T)


── Utilisation ratio distribution ───────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
feedin_utilisation_ratio,338.0000,0.4717,0.3795,0.0000,0.2010,0.4188,0.6318,2.0000
consumption_utilisation_ratio,357.0000,30812325.6758,531747871.5545,0.0682,0.5743,0.7719,0.9168,10000000000.0000


In [9]:
# How many areas are already over-subscribed (ratio > 1)?
over_feedin = (vg["feedin_utilisation_ratio"] > 1).sum()
over_cons   = (vg["consumption_utilisation_ratio"] > 1).sum()
total       = len(vg)

print(f"Over-subscribed feed-in areas        : {over_feedin:3d} / {total}  "
      f"({over_feedin/total:.0%})")
print(f"Over-subscribed consumption areas    : {over_cons:3d} / {total}  "
      f"({over_cons/total:.0%})")
print()
print("Top 10 areas by consumption utilisation:")
display(
    vg[["supply_area_id", "regional_grid_operator",
        "consumption_utilisation_ratio", "feedin_utilisation_ratio"]]
    .sort_values("consumption_utilisation_ratio", ascending=False)
    .head(10)
    .reset_index(drop=True)
)


Over-subscribed feed-in areas        :  31 / 608  (5%)
Over-subscribed consumption areas    :  55 / 608  (9%)

Top 10 areas by consumption utilisation:


,supply_area_id,regional_grid_operator,consumption_utilisation_ratio,feedin_utilisation_ratio
0,OS WEESP 10-9i,Liander,10000000000.0000,0.2500
1,Medemblik Transformator 1,Liander,1000000000.0000,0.3500
2,OS BERGUM CENTRALE 10-2i,Liander,1.8824,1.4118
3,OS IJPOLDER 10-4i,Liander,1.5937,0.5000
4,OS HAARLEMMERMEER 20-2i,Liander,1.5795,0.0000
5,CM29/45,Stedin,1.5722,NaN
6,OS HOOFDDORP 10-2i,Liander,1.4211,0.2500
7,OS AALSMEER BLOEMENVEILING 10-1i,Liander,1.4167,0.5556
8,CM58,Stedin,1.4000,NaN
9,CM57,Stedin,1.4000,NaN


---
## 5 · Metric 2 — Queue Intensity Ratios

### Formula
$$\text{queue\_intensity} = \frac{\text{queue\_MW}}{\text{available\_capacity\_MW}}$$

### Why queue depth matters

The queue tells you how many MW of connection requests are **waiting in line** because the grid is full. A high queue intensity means:

1. **Persistence:** The opportunity won't vanish after one small upgrade — the backlog refills.
2. **DSO urgency:** A large queue represents real economic pressure. Companies waiting for connections are losing revenue; DSOs face regulatory scrutiny. They are *motivated to pay* for flexibility solutions.
3. **Market depth:** Many MW in the queue = large total addressable market for aggregators.

> Example: Queue = 200 MW, Available = 50 MW → intensity = 4.0 — the queue is **four times** the existing grid capacity. This area will remain congested for years regardless of moderate upgrades.


In [10]:
vg["feedin_queue_intensity"] = (
    vg["feedin_queue_mw"] /
    (vg["available_feedin_capacity_mw"] + EPSILON)
)

vg["consumption_queue_intensity"] = (
    vg["consumption_queue_mw"] /
    (vg["available_consumption_capacity_mw"] + EPSILON)
)

print("── Queue intensity distribution ─────────────────────────────────────────")
display(vg[["feedin_queue_intensity", "consumption_queue_intensity"]].describe().T)


── Queue intensity distribution ─────────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
feedin_queue_intensity,255.0000,0.2304,0.2747,0.0000,0.0607,0.1256,0.2955,1.8082
consumption_queue_intensity,342.0000,4762924.2369,88081905.7553,0.0000,0.0959,0.2017,0.3619,1628920000.0000


In [11]:
print("Top 10 areas by total queue intensity (feed-in + consumption):")
vg["total_queue_intensity"] = vg["feedin_queue_intensity"] + vg["consumption_queue_intensity"]
display(
    vg[["supply_area_id", "regional_grid_operator",
        "feedin_queue_intensity", "consumption_queue_intensity", "total_queue_intensity"]]
    .sort_values("total_queue_intensity", ascending=False)
    .head(10)
    .reset_index(drop=True)
)


Top 10 areas by total queue intensity (feed-in + consumption):


,supply_area_id,regional_grid_operator,feedin_queue_intensity,consumption_queue_intensity,total_queue_intensity
0,OS ULFT 20-3i,Liander,1.8082,0.4848,2.2930
1,MZ,Enexis,1.5217,0.7412,2.2629
2,VENR,Enexis,1.6421,0.3628,2.0049
3,ISM,Enexis,1.3206,0.6004,1.9210
4,OS ALPHEN WEST 10-1i,Liander,0.5079,1.3705,1.8784
5,OS OOSTERHOUT 20-2i,Liander,0.5067,1.1866,1.6933
6,HPS,Enexis,1.1496,0.4976,1.6472
7,OS HOOFDDORP 10-1i,Liander,0.3577,1.2764,1.6341
8,ZLH,Enexis,1.0657,0.5061,1.5719
9,OS HARSELAAR 20-3i,Liander,0.7061,0.8515,1.5576


---
## 6 · Metric 3 — Absolute Capacity Gap (MW)

### Formula
$$\text{capacity\_gap\_MW} = \text{required\_MW} - \text{available\_MW}$$

### Why absolute MW gap complements the ratios

The utilisation ratio tells you *how stressed* an area is proportionally. The absolute gap tells you *how big the market is* in physical terms.

- A ratio of 3.0 on a 5 MW area = 10 MW gap → niche opportunity  
- A ratio of 1.5 on a 200 MW area = 100 MW gap → major market

You want **both** high ratio (structural problem) and large absolute gap (big enough to be commercially interesting). The composite score in Section 8 captures this by using both dimensions.


In [12]:
vg["feedin_capacity_gap_mw"] = (
    vg["required_feedin_capacity_mw"] - vg["available_feedin_capacity_mw"]
)
vg["consumption_capacity_gap_mw"] = (
    vg["required_consumption_capacity_mw"] - vg["available_consumption_capacity_mw"]
)
vg["total_queue_mw"] = vg["feedin_queue_mw"] + vg["consumption_queue_mw"]

print("── Capacity gap distribution (MW) ───────────────────────────────────────")
display(vg[["feedin_capacity_gap_mw", "consumption_capacity_gap_mw",
            "total_queue_mw"]].describe().T)


── Capacity gap distribution (MW) ───────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
feedin_capacity_gap_mw,338.0000,-44.5517,40.3974,-219.3547,-64.7500,-38.0000,-18.0000,130.1670
consumption_capacity_gap_mw,357.0000,-23.3107,38.1808,-323.8190,-27.0267,-12.0000,-3.0000,51.0000
total_queue_mw,471.0000,25.7237,41.4568,0.0000,0.0000,9.6751,31.9880,395.4780


In [13]:
# Scatter: utilisation ratio vs absolute gap → quadrant view
# Top-right quadrant = highest priority (high ratio AND large gap)
print("Top 15 areas: high consumption ratio AND large consumption gap")
display(
    vg[vg["consumption_capacity_gap_mw"] > 0][[
        "supply_area_id", "regional_grid_operator",
        "consumption_utilisation_ratio", "consumption_capacity_gap_mw",
        "consumption_queue_mw"
    ]]
    .sort_values(["consumption_utilisation_ratio", "consumption_capacity_gap_mw"],
                 ascending=False)
    .head(15)
    .reset_index(drop=True)
)


Top 15 areas: high consumption ratio AND large consumption gap


,supply_area_id,regional_grid_operator,consumption_utilisation_ratio,consumption_capacity_gap_mw,consumption_queue_mw
0,OS WEESP 10-9i,Liander,10000000000.0000,10.0000,NaN
1,Medemblik Transformator 1,Liander,1000000000.0000,1.0000,1.6289
2,OS BERGUM CENTRALE 10-2i,Liander,1.8824,15.0000,3.8120
3,OS IJPOLDER 10-4i,Liander,1.5937,19.0000,19.4204
4,OS HAARLEMMERMEER 20-2i,Liander,1.5795,51.0000,NaN
5,CM29/45,Stedin,1.5722,21.0000,1.6370
6,OS HOOFDDORP 10-2i,Liander,1.4211,8.0000,12.9588
7,OS AALSMEER BLOEMENVEILING 10-1i,Liander,1.4167,15.0000,22.9978
8,CM57,Stedin,1.4000,10.8000,12.1900
9,CM58,Stedin,1.4000,10.8000,2.5260


---
## 7 · Metric 4 — Grid Stress Index (log-scaled composite)

### Formula
$$\text{grid\_stress\_index} = \ln\left(1 + \frac{\text{feedin\_util} + \text{consumption\_util}}{2}\right)$$

### Why log-scale?

Raw utilisation ratios can span a very wide range (0.1 to 50+). Without compression:
- An area with ratio 50 would dominate the scoring 5× over an area with ratio 10, even though both are severely congested and commercially attractive.
- `log1p` compresses extreme values while preserving rank order. It rewards congestion but avoids a single extreme outlier monopolising the top of the ranked list.

This gives us a single **"how stressed is this area overall?"** number that is both balanced and interpretable.


In [14]:
vg["grid_stress_index"] = np.log1p(
    (vg["feedin_utilisation_ratio"] + vg["consumption_utilisation_ratio"]) / 2
)

print("── Grid stress index distribution ───────────────────────────────────────")
display(vg["grid_stress_index"].describe().to_frame().T)

print()
print("Top 10 most stressed supply areas:")
display(
    vg[["supply_area_id", "regional_grid_operator", "grid_stress_index",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio"]]
    .sort_values("grid_stress_index", ascending=False)
    .head(10)
    .reset_index(drop=True)
)


── Grid stress index distribution ───────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
grid_stress_index,324.0000,0.5774,1.6342,0.1075,0.3584,0.4373,0.5342,22.3327



Top 10 most stressed supply areas:


,supply_area_id,regional_grid_operator,grid_stress_index,feedin_utilisation_ratio,consumption_utilisation_ratio
0,OS WEESP 10-9i,Liander,22.3327,0.2500,10000000000.0000
1,Medemblik Transformator 1,Liander,20.0301,0.3500,1000000000.0000
2,OS BERGUM CENTRALE 10-2i,Liander,0.9734,1.4118,1.8824
3,CM56/78,Stedin,0.9192,2.0000,1.0146
4,HGLB,Enexis,0.7760,1.4093,0.9360
5,OS ZEVENHUIZEN 10-2i,Liander,0.7732,1.4333,0.9000
6,CM34/46,Stedin,0.7542,1.1159,1.1361
7,CM29/64,Stedin,0.7517,1.2446,0.9968
8,CM35/72,Stedin,0.7492,1.3901,0.8404
9,OMD,Enexis,0.7410,1.2306,0.9655


---
## 8 · Project Relief Discount

### Rationale

The `projecten` table lists planned grid upgrades. If a substation expansion is scheduled for next year, **the congestion window is temporary** — a DSO has little incentive to sign a 3-year flexibility contract when their engineers will resolve the bottleneck in 12 months.

We therefore **discount** the opportunity score for areas where near-term relief is already programmed:

- Each project completing within **24 months** reduces the score by **15%**
- The floor is clamped at **10%** — even heavily-relieved areas retain residual value (construction periods, transition contracts)
- Areas with **zero planned projects** receive **no discount** — these are the most attractive long-term targets


In [16]:
# proj["completion_date"] = pd.to_datetime(proj["completion_date"], errors="coerce")


proj["completion_date"] = pd.to_datetime(
    proj["einddatum_string"],
    format="%Y",
    errors="coerce"
)

NEAR_TERM_HORIZON = pd.Timestamp.today() + pd.DateOffset(years=2)

# Count near-term projects per area
near_term = (
    proj[proj["completion_date"] <= NEAR_TERM_HORIZON]
    .groupby("area_id")
    .size()
    .reset_index(name="near_term_project_count")
)

all_proj = (
    proj.groupby("area_id")
    .size()
    .reset_index(name="total_project_count")
)

project_relief = all_proj.merge(near_term, on="area_id", how="left")
project_relief["near_term_project_count"] = (
    project_relief["near_term_project_count"].fillna(0).astype(int)
)

print("── Project relief summary ───────────────────────────────────────────────")
display(project_relief.describe().T)
print()
print("Areas with most near-term projects (highest relief discount):")
display(
    project_relief.sort_values("near_term_project_count", ascending=False).head(10)
)


── Project relief summary ───────────────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
total_project_count,346.0000,2.7312,3.3914,1.0000,1.0000,2.0000,3.0000,28.0000
near_term_project_count,346.0000,0.5289,1.2185,0.0000,0.0000,0.0000,1.0000,8.0000



Areas with most near-term projects (highest relief discount):


,area_id,total_project_count,near_term_project_count
119,Noord-Brabant,18,8
120,Noord-Holland,27,8
33,CM37,14,8
26,CM22,28,8
40,CM48,16,6
286,TH058_23,8,6
282,TH010_10,4,4
292,TH212_10_AB,4,4
289,TH201_10_A,8,4
98,Haven van Rotterdam,8,4


In [17]:
# Join project counts to supply-area table
vg = vg.merge(
    project_relief.rename(columns={"area_id": "supply_area_id"}),
    on="supply_area_id",
    how="left"
)
vg["total_project_count"]     = vg["total_project_count"].fillna(0).astype(int)
vg["near_term_project_count"] = vg["near_term_project_count"].fillna(0).astype(int)

# Discount factor: 1.0 = no discount, min = 0.10
vg["relief_discount"] = np.clip(
    1.0 - (vg["near_term_project_count"] * 0.15),
    a_min=0.1, a_max=1.0
)

print("── Relief discount distribution ─────────────────────────────────────────")
display(vg["relief_discount"].value_counts().sort_index(ascending=False))


── Relief discount distribution ─────────────────────────────────────────


relief_discount
1.0000    529
0.8500     40
0.7000     30
0.5500      1
0.4000      4
0.1000      2
0.1000      2
Name: count, dtype: int64

---
## 9 · Composite Valorization Score

### Architecture

$$\text{valorization\_score} = \left(\sum_{i} w_i \cdot \tilde{x}_i\right) \times \text{relief\_discount}$$

where $\tilde{x}_i$ is the min-max normalised value of component $i$.

### Weight rationale

| Component | Weight | Reason |
|---|---|---|
| `grid_stress_index` | **0.30** | Gating criterion — no stress = no market |
| `consumption_utilisation_ratio` | **0.25** | Demand-response commands premium on EPEX evening peak (18:00–20:00 CET) |
| `feedin_utilisation_ratio` | **0.15** | Curtailment contracts exist but at lower margins |
| `feedin_queue_intensity` | **0.15** | Large feed-in queue → DSO has budget committed for solutions |
| `consumption_queue_intensity` | **0.15** | Large consumption queue → industrial players are actively seeking connection |

**Why min-max normalise first?**  
Without normalisation, a metric measured in hundreds of MW would dominate metrics measured as ratios. Scaling all components to [0, 1] ensures each weight truly controls the contribution, regardless of original units or magnitude.


In [18]:
WEIGHTS = {
    "grid_stress_index":             0.30,
    "consumption_utilisation_ratio": 0.25,
    "feedin_utilisation_ratio":      0.15,
    "feedin_queue_intensity":        0.15,
    "consumption_queue_intensity":   0.15,
}

def minmax_scale(series: pd.Series) -> pd.Series:
    """Scale a series to [0, 1]. Returns 0.5 for a constant series."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

# Build weighted score components
score_parts = pd.DataFrame(index=vg.index)
for col, weight in WEIGHTS.items():
    norm_col = f"{col}_norm"
    score_parts[norm_col] = minmax_scale(vg[col]) * weight
    print(f"  {col:<38s}  weight={weight:.2f}  "
          f"normalised range: [{score_parts[norm_col].min():.3f}, "
          f"{score_parts[norm_col].max():.3f}]")

vg["raw_valorization_score"] = score_parts.sum(axis=1)
vg["valorization_score"]     = vg["raw_valorization_score"] * vg["relief_discount"]

vg["valorization_rank"] = vg["valorization_score"].rank(
    ascending=False, method="min"
).astype(int)

print()
print("── Score distribution ───────────────────────────────────────────────────")
display(vg[["raw_valorization_score", "valorization_score"]].describe().T)


  grid_stress_index                       weight=0.30  normalised range: [0.000, 0.300]
  consumption_utilisation_ratio           weight=0.25  normalised range: [0.000, 0.250]
  feedin_utilisation_ratio                weight=0.15  normalised range: [0.000, 0.150]
  feedin_queue_intensity                  weight=0.15  normalised range: [0.000, 0.150]
  consumption_queue_intensity             weight=0.15  normalised range: [0.000, 0.150]

── Score distribution ───────────────────────────────────────────────────


,count,mean,std,min,25%,50%,75%,max
raw_valorization_score,608.0000,0.0318,0.0502,0.0000,0.0000,0.0071,0.0491,0.5688
valorization_score,608.0000,0.0307,0.0490,0.0000,0.0000,0.0071,0.0480,0.5688


### 📊 Score Distribution Diagnostic

Before assigning tiers, always inspect the actual percentile breakpoints of `valorization_score`.

**Why this matters:**  
Min-max normalisation guarantees the score range is [0, 1], but says nothing about the *shape* inside that range. In congestion datasets, most areas have low scores (they are not congested), while a small number of genuinely stressed areas pull the right tail. Fixed absolute bins like `[0, 0.2, 0.4, 0.6, 0.8, 1]` are calibrated for a **uniform** distribution — they collapse to Tier 5 for any right-skewed distribution like this one.

The cell below prints the actual percentile thresholds so you can see where the real breaks are.


In [19]:
s = vg["valorization_score"]
percentiles = [0, 10, 25, 50, 60, 70, 80, 90, 95, 99, 100]
cuts = {f"p{p:03d}": s.quantile(p / 100) for p in percentiles}

print("valorization_score — percentile breakdown:")
print(f"  {'Percentile':<14} {'Score':>10}")
print("  " + "-" * 26)
for pct, val in cuts.items():
    label = f"p{pct[1:]}  (top {100 - int(pct[1:]):>3}%)"
    print(f"  {label:<22} {val:>10.6f}")

print()
print(f"  mean   : {s.mean():.6f}")
print(f"  median : {s.median():.6f}")
print(f"  skew   : {s.skew():.3f}  (> 1 = right-skewed → quantile tiers required)")


valorization_score — percentile breakdown:
  Percentile          Score
  --------------------------
  p000  (top 100%)         0.000000
  p010  (top  90%)         0.000000
  p025  (top  75%)         0.000000
  p050  (top  50%)         0.007072
  p060  (top  40%)         0.024949
  p070  (top  30%)         0.041666
  p080  (top  20%)         0.058496
  p090  (top  10%)         0.087201
  p095  (top   5%)         0.107547
  p099  (top   1%)         0.169109
  p100  (top   0%)         0.568750

  mean   : 0.030691
  median : 0.007072
  skew   : 4.298  (> 1 = right-skewed → quantile tiers required)


### ✅ Quantile-Based Tier Assignment

**Root cause of the collapsed tiers:** Fixed absolute bins assume scores are uniformly distributed. In this dataset the vast majority of supply areas are not congested, so ~99% of scores cluster near 0 — all landing in Tier 5 under the old scheme.

**Fix: percentile-based (quantile) cuts.** These are defined relative to the actual distribution, so they **always produce a meaningful spread** regardless of skew:

| Tier | Percentile band | Interpretation |
|------|----------------|----------------|
| Tier 1 — Priority | top 10% (≥ p90) | Highest-stress areas — immediate commercial targets |
| Tier 2 — High | p75 – p90 | Strong candidates for mid-term contracts |
| Tier 3 — Moderate | p50 – p75 | Worth monitoring; approaching threshold |
| Tier 4 — Low | p25 – p50 | Below median; low urgency |
| Tier 5 — Minimal | bottom 25% (< p25) | No congestion signal; not actionable now |

> **Note:** With quantile tiers, the tier boundaries shift each time the dataset is refreshed (new DSO data releases). That is correct behaviour — the tiers always reflect the *relative* position within the current snapshot, which is what matters for prioritisation.


In [20]:
# --- Quantile-based tier boundaries -------------------------------------------
# Defined on the actual score distribution, not arbitrary absolute thresholds.
# This guarantees a meaningful spread even for heavily right-skewed distributions.
q25, q50, q75, q90 = (
    vg["valorization_score"].quantile(0.25),
    vg["valorization_score"].quantile(0.50),
    vg["valorization_score"].quantile(0.75),
    vg["valorization_score"].quantile(0.90),
)

print(f"Tier boundaries derived from score distribution:")
print(f"  Tier 1 — Priority  : score ≥ {q90:.6f}  (top 10%)")
print(f"  Tier 2 — High      : {q75:.6f} ≤ score < {q90:.6f}  (p75–p90)")
print(f"  Tier 3 — Moderate  : {q50:.6f} ≤ score < {q75:.6f}  (p50–p75)")
print(f"  Tier 4 — Low       : {q25:.6f} ≤ score < {q50:.6f}  (p25–p50)")
print(f"  Tier 5 — Minimal   : score <  {q25:.6f}  (bottom 25%)")
print()

vg["opportunity_tier"] = pd.cut(
    vg["valorization_score"],
    bins=[-np.inf, q25, q50, q75, q90, np.inf],
    labels=[
        "Tier 5 — Minimal",
        "Tier 4 — Low",
        "Tier 3 — Moderate",
        "Tier 2 — High",
        "Tier 1 — Priority",
    ],
)

tier_counts = vg["opportunity_tier"].value_counts().sort_index()
print("── Tier distribution ────────────────────────────────────────────────────")
for tier, count in tier_counts.items():
    pct = count / len(vg) * 100
    bar = "█" * int(pct / 2)
    print(f"  {str(tier):<22}  {count:>4} areas  ({pct:5.1f}%)  {bar}")


Tier boundaries derived from score distribution:
  Tier 1 — Priority  : score ≥ 0.087201  (top 10%)
  Tier 2 — High      : 0.047989 ≤ score < 0.087201  (p75–p90)
  Tier 3 — Moderate  : 0.007072 ≤ score < 0.047989  (p50–p75)
  Tier 4 — Low       : 0.000000 ≤ score < 0.007072  (p25–p50)
  Tier 5 — Minimal   : score <  0.000000  (bottom 25%)

── Tier distribution ────────────────────────────────────────────────────
  Tier 5 — Minimal         235 areas  ( 38.7%)  ███████████████████
  Tier 4 — Low              69 areas  ( 11.3%)  █████
  Tier 3 — Moderate        152 areas  ( 25.0%)  ████████████
  Tier 2 — High             91 areas  ( 15.0%)  ███████
  Tier 1 — Priority         61 areas  ( 10.0%)  █████


In [21]:
print("── TOP 20 SUPPLY AREAS BY VALORIZATION SCORE ────────────────────────────")
display(
    vg[[
        "valorization_rank", "supply_area_id", "regional_grid_operator",
        "valorization_score", "opportunity_tier",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio",
        "feedin_queue_intensity", "consumption_queue_intensity",
        "total_queue_mw", "near_term_project_count", "relief_discount",
    ]]
    .sort_values("valorization_rank")
    .head(20)
    .reset_index(drop=True)
)


── TOP 20 SUPPLY AREAS BY VALORIZATION SCORE ────────────────────────────


,valorization_rank,supply_area_id,regional_grid_operator,valorization_score,opportunity_tier,feedin_utilisation_ratio,consumption_utilisation_ratio,feedin_queue_intensity,consumption_queue_intensity,total_queue_mw,near_term_project_count,relief_discount
0,1,OS WEESP 10-9i,Liander,0.5688,Tier 1 — Priority,0.2500,10000000000.0000,NaN,NaN,NaN,0,1.0000
1,2,Medemblik Transformator 1,Liander,0.4702,Tier 1 — Priority,0.3500,1000000000.0000,NaN,1628920000.0000,NaN,0,1.0000
2,3,OS ULFT 20-3i,Liander,0.2679,Tier 1 — Priority,1.4783,0.2826,1.8082,0.4848,105.4766,0,1.0000
3,4,MZ,Enexis,0.2351,Tier 1 — Priority,1.3508,0.5570,1.5217,0.7412,188.6460,0,1.0000
4,5,VENR,Enexis,0.1870,Tier 1 — Priority,0.6173,0.4785,1.6421,0.3628,395.4780,0,1.0000
5,6,ISM,Enexis,0.1846,Tier 1 — Priority,0.9342,0.2978,1.3206,0.6004,83.8010,0,1.0000
6,7,VO,Enexis,0.1697,Tier 1 — Priority,1.2026,0.8682,0.8601,0.5108,32.8190,0,1.0000
7,8,ZLH,Enexis,0.1615,Tier 1 — Priority,0.8848,0.7744,1.0657,0.5061,142.6160,0,1.0000
8,9,CM56/78,Stedin,0.1610,Tier 1 — Priority,2.0000,1.0146,NaN,0.0000,NaN,0,1.0000
9,10,CM08,Stedin,0.1586,Tier 1 — Priority,1.6200,NaN,0.4475,NaN,8.4120,0,1.0000


---
## 10 · Map Scores to Postcode Level (`pc6`)

The valorization scores live at the supply-area level. We join them down to the postcode grain so that commercial teams can work with the familiar 6-digit postcode format for:
- Field sales routing
- Direct-mail or email campaigns
- CRM territory assignment

We also compute a **`postcode_congestion_severity`** score (0–6) by summing the two color codes. This acts as a **tie-breaker** within the same supply area: if two postcodes share the same supply-area score, the one with the more severe local congestion coloring gets ranked higher.

> Color code mapping: `−1` = no info (treated as 0), `0` = transparent, `1` = yellow, `2` = orange, `3` = red


In [22]:
pc6 = pc6.merge(
    vg[[
        "supply_area_id", "valorization_score", "valorization_rank",
        "opportunity_tier", "grid_stress_index",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio",
        "feedin_queue_intensity", "consumption_queue_intensity",
        "total_queue_mw", "feedin_capacity_gap_mw", "consumption_capacity_gap_mw",
        "near_term_project_count", "relief_discount",
    ]],
    on="supply_area_id",
    how="left"
)

# Clip -1 (no info) to 0 before summing
pc6["consumption_color_clean"] = pc6["consumption_color"].clip(lower=0)
pc6["generation_color_clean"]  = pc6["generation_color"].clip(lower=0)
pc6["postcode_congestion_severity"] = (
    pc6["consumption_color_clean"] + pc6["generation_color_clean"]
)

# Sort: highest valorization score first; within same score, most severe color first
pc6 = pc6.sort_values(
    ["valorization_score", "postcode_congestion_severity"],
    ascending=[False, False]
).reset_index(drop=True)

pc6["postcode_rank"] = pc6.index + 1

print(f"✓ Postcode table enriched: {pc6.shape[0]:,} rows")
display(pc6.head(5)[[
    "postcode_rank", "postcode", "supply_area_name", "regional_grid_operator",
    "consumption_color", "generation_color", "postcode_congestion_severity",
    "valorization_score", "opportunity_tier"
]])


✓ Postcode table enriched: 462,606 rows


,postcode_rank,postcode,supply_area_name,regional_grid_operator,consumption_color,generation_color,postcode_congestion_severity,valorization_score,opportunity_tier
0,1,1398XT,OS WEESP 10-9i,Liander,3,0,3,0.5688,Tier 1 — Priority
1,2,1398XR,OS WEESP 10-9i,Liander,3,0,3,0.5688,Tier 1 — Priority
2,3,1398XP,OS WEESP 10-9i,Liander,3,0,3,0.5688,Tier 1 — Priority
3,4,1398XN,OS WEESP 10-9i,Liander,3,0,3,0.5688,Tier 1 — Priority
4,5,1398XM,OS WEESP 10-9i,Liander,3,0,3,0.5688,Tier 1 — Priority


---
## 11 · Regional Aggregation & Summary Statistics

We aggregate from postcode level up to supply area, computing:

- **`mean_congestion_severity`** — average congestion across postcodes in the region
- **`std_congestion_severity`** — low std dev = uniform congestion; high std dev = isolated hotspots
- **`pct_red_consumption`** — share of postcodes at maximum congestion (red) for demand
- **`pct_red_generation`** — same for feed-in

> 🔑 **Key insight:** A region where `pct_red > 0.7` and `std_dev` is low is *systemically* congested, not just a local anomaly. This is the most attractive target for framework flexibility agreements at DSO level.


In [23]:
pc6_agg = (
    pc6.groupby(["supply_area_id", "supply_area_name", "regional_grid_operator"])
    .agg(
        postcode_count             = ("postcode", "count"),
        mean_congestion_severity   = ("postcode_congestion_severity", "mean"),
        median_congestion_severity = ("postcode_congestion_severity", "median"),
        std_congestion_severity    = ("postcode_congestion_severity", "std"),
        pct_red_consumption        = ("consumption_color_clean", lambda x: (x == 3).mean()),
        pct_red_generation         = ("generation_color_clean",  lambda x: (x == 3).mean()),
    )
    .reset_index()
)

regional_summary = pc6_agg.merge(
    vg[[
        "supply_area_id", "valorization_score", "valorization_rank",
        "opportunity_tier", "grid_stress_index",
        "feedin_utilisation_ratio", "consumption_utilisation_ratio",
        "total_queue_mw", "feedin_capacity_gap_mw", "consumption_capacity_gap_mw",
        "near_term_project_count", "relief_discount",
    ]],
    on="supply_area_id",
    how="left"
).sort_values("valorization_rank")

print("── TOP 15 SUPPLY AREAS — REGIONAL SUMMARY ───────────────────────────────")
display(
    regional_summary[[
        "valorization_rank", "supply_area_name", "regional_grid_operator",
        "postcode_count", "valorization_score", "opportunity_tier",
        "mean_congestion_severity", "std_congestion_severity",
        "pct_red_consumption", "pct_red_generation",
        "total_queue_mw",
    ]].head(15).reset_index(drop=True)
)


── TOP 15 SUPPLY AREAS — REGIONAL SUMMARY ───────────────────────────────


,valorization_rank,supply_area_name,regional_grid_operator,postcode_count,valorization_score,opportunity_tier,mean_congestion_severity,std_congestion_severity,pct_red_consumption,pct_red_generation,total_queue_mw
0,1,OS WEESP 10-9i,Liander,182,0.5688,Tier 1 — Priority,3.0000,0.0000,1.0000,0.0000,NaN
1,2,Medemblik Transformator 1,Liander,7,0.4702,Tier 1 — Priority,5.0000,0.0000,1.0000,0.0000,NaN
2,3,OS ULFT 20-3i,Liander,240,0.2679,Tier 1 — Priority,2.1250,0.4851,0.0000,0.0000,105.4766
3,4,MZ Maarheeze,Enexis,1302,0.2351,Tier 1 — Priority,4.0000,0.0000,0.0000,1.0000,188.6460
4,5,VENR Venray,Enexis,1906,0.1870,Tier 1 — Priority,5.2602,1.9732,0.8767,0.8767,395.4780
5,6,ISM IJsselmuiden,Enexis,321,0.1846,Tier 1 — Priority,3.0000,0.0000,1.0000,0.0000,83.8010
6,7,VO Veenoord,Enexis,675,0.1697,Tier 1 — Priority,6.0000,0.0000,1.0000,1.0000,32.8190
7,8,ZLH Zwolle Hessenweg,Enexis,544,0.1615,Tier 1 — Priority,1.0000,0.0000,0.0000,0.0000,142.6160
8,9,Stompwijk en Leidschendam-Zuid,Stedin,131,0.1610,Tier 1 — Priority,6.0000,0.0000,1.0000,1.0000,NaN
9,10,Waddinxveen,Stedin,720,0.1586,Tier 1 — Priority,3.0000,0.0000,0.0000,0.0000,8.4120


---
## 12 · Operator Roll-Up — Strategic Account View

Aggregate from supply areas to the regional grid operator (DSO) level.

This answers: **"Which DSO manages the most congested grid footprint?"**

Prioritising which DSO relationship to build first is a strategic decision — a framework flexibility agreement with a single DSO that manages 40% of Tier 1 areas is far more efficient than individual area-by-area contracts.

Key metric: **`priority_areas`** = count of Tier 1 supply areas under this operator.


In [24]:
operator_summary = (
    regional_summary.groupby("regional_grid_operator")
    .agg(
        supply_area_count        = ("supply_area_id", "count"),
        total_postcode_count     = ("postcode_count", "sum"),
        mean_valorization_score  = ("valorization_score", "mean"),
        median_valorization_score= ("valorization_score", "median"),
        std_valorization_score   = ("valorization_score", "std"),
        total_queue_mw           = ("total_queue_mw", "sum"),
        priority_areas           = ("opportunity_tier",
                                    lambda x: (x == "Tier 1 — Priority").sum()),
    )
    .reset_index()
    .sort_values("mean_valorization_score", ascending=False)
)

print("── OPERATOR ROLL-UP ─────────────────────────────────────────────────────")
display(operator_summary.reset_index(drop=True))


── OPERATOR ROLL-UP ─────────────────────────────────────────────────────


,regional_grid_operator,supply_area_count,total_postcode_count,mean_valorization_score,median_valorization_score,std_valorization_score,total_queue_mw,priority_areas
0,Westland,2,3351,0.0830,0.0830,0.0303,104.0000,1
1,Enexis,118,159640,0.0643,0.0568,0.0427,7117.5100,29
2,Coteq,4,3270,0.0619,0.0681,0.0147,130.3860,0
3,Rendo,2,1913,0.0612,0.0612,0.0084,98.7320,0
4,Liander,244,177298,0.0368,0.0265,0.0560,3249.6057,18
5,Stedin,208,117098,0.0089,0.0000,0.0310,1488.5760,13


---
## 13 · Final Outputs

Three deliverables are exported as CSV files in the `outputs/` folder:

| File | Description | Primary user |
|---|---|---|
| `ranked_postcodes.csv` | All postcodes ranked by score | Field sales, CRM upload |
| `ranked_supply_areas.csv` | Supply areas with full metrics | Energy market team |
| `ranked_operators.csv` | DSO roll-up | Strategic accounts / BD |


In [25]:
import os
os.makedirs("outputs", exist_ok=True)

# Ranked postcode list — primary commercial deliverable
pc6_export_cols = [
    "postcode_rank", "postcode", "supply_area_id", "supply_area_name",
    "regional_grid_operator", "tennet_id",
    "consumption_color", "generation_color", "postcode_congestion_severity",
    "valorization_score", "opportunity_tier",
    "feedin_utilisation_ratio", "consumption_utilisation_ratio",
    "feedin_queue_intensity", "consumption_queue_intensity",
    "total_queue_mw", "near_term_project_count",
]

pc6[pc6_export_cols].to_excel("outputs/ranked_postcodes.xlsx", index=False)
regional_summary.to_excel("outputs/ranked_supply_areas.xlsx", index=False)
operator_summary.to_excel("outputs/ranked_operators.xlsx", index=False)

print("✓ outputs/ranked_postcodes.xlsx")
print("✓ outputs/ranked_supply_areas.xlsx")
print("✓ outputs/ranked_operators.xlsx")


✓ outputs/ranked_postcodes.xlsx
✓ outputs/ranked_supply_areas.xlsx
✓ outputs/ranked_operators.xlsx


---
## 14 · Interpreting Results — What Does a High Score Look Like?

### Profile of a Tier 1 — Priority postcode/area

| Signal | Threshold | Meaning |
|---|---|---|
| `consumption_utilisation_ratio` | ≥ 1.5 | Grid at 150 %+ of what it can handle |
| `feedin_queue_intensity` | ≥ 2.0 | Queue is 2× the existing capacity |
| `consumption_color` | = 3 (red) | Maximum local congestion coloring |
| `near_term_project_count` | = 0 | No relief scheduled — long market window |
| `valorization_score` | ≥ 0.80 | Top tier composite |

### Beware of false positives

A postcode may look red on the capacity map but have `near_term_project_count = 3` — the DSO has already secured budget and contractors. The relief discount correctly **demotes** these so your sales team doesn't spend cycles on fading opportunities.

### Commercial translation

In the Dutch flexibility market (GOPACS / CODS), certified flexibility in a congested area commands:
- **€40,000 – €150,000 / MW / year** for day-ahead flexibility contracts
- Higher rates for intra-day and emergency dispatch

A Tier 1 area with `total_queue_mw = 100` and no near-term relief represents a **potential market of €4M – €15M / year** for an aggregator operating in that area.


---
## 15 · Prioritize Within Tier 1 — Top 10 Regions + 4-Digit Postcode Zones

We have 61 Tier 1 supply areas and ~54,000 postcodes. Two steps to cut this down:

1. **Pick the top 10 supply areas** by `valorization_score` within Tier 1  
2. **Collapse 6-digit postcodes → 4-digit** (`5831SK` → `5831`) — these map to  
   recognisable industrial/commercial zones and are the standard grain for  
   B2B prospecting databases (KvK, Dun & Bradstreet, etc.)  
3. **Rank 4-digit zones** within each top-10 area by:
   - Mean `valorization_score` inherited from their pc6 postcodes  
   - Share of postcodes at maximum congestion (`pct_red`)  
   - Postcode density (proxy for industrial activity volume)

The output is a focused list of **4-digit postcode zones per region**,  
ready to feed into a company search (KvK API, OpenStreetMap Overpass, etc.).


In [46]:
import pandas as pd
import numpy as np
# from sklearn.preprocessing import MinMaxScaler # fallback _mm is used below

# ── NEW: Load Industry Relevance Data -----------------------------------------
# Load the CSV generated previously
industry_df = pd.read_csv("raw_data/netherlands_prospection_energy_intensive.csv")

# Standardize columns to lowercase to prevent capitalization KeyErrors permanently
industry_df.columns = industry_df.columns.str.lower()

industry_df["postal code"] = industry_df["postal code"].astype(str)

# Extract the numeric rank (1 to 10) from the string (e.g., "1 - Extremely High" -> 1)
industry_df["relevance_num"] = industry_df["relevance"].str.extract(r'^(\d+)').astype(float)

# Invert it so higher is better for the ML scaler (Rank 1 becomes Score 10, Rank 10 becomes Score 1)
industry_df["industry_score"] = 11 - industry_df["relevance_num"]


# ── Step 1: Top 10 Tier 1 supply areas ----------------------------------------
top10_areas = (
    vg[vg["opportunity_tier"] == "Tier 1 — Priority"]
    .sort_values("valorization_score", ascending=False)
    .head(10)
    [["supply_area_id", "regional_grid_operator", 
      "valorization_score", "valorization_rank",
      "consumption_utilisation_ratio", "feedin_utilisation_ratio", 
      "total_queue_mw", "near_term_project_count"]]
    .reset_index(drop=True)
)
top10_areas.index += 1  # 1-based rank
print("TOP 10 TIER-1 SUPPLY AREAS")
display(top10_areas)


# ── Step 2: Filter pc6 to those areas + collapse to pc4 -----------------------
top10_ids = top10_areas["supply_area_id"].tolist()

pc4 = (
    pc6[pc6["supply_area_id"].isin(top10_ids)]
    .assign(pc4=lambda df: df["postcode"].str[:4])   # "5831SK" → "5831"
    .groupby(["supply_area_id", "supply_area_name", "regional_grid_operator", "pc4"])
    .agg(
        pc6_count          = ("postcode",                    "count"),   # density proxy
        mean_val_score     = ("valorization_score",          "mean"),
        max_val_score      = ("valorization_score",          "max"),
        mean_congestion    = ("postcode_congestion_severity","mean"),
        pct_red_consumption= ("consumption_color_clean",     lambda x: (x == 3).mean()),
        pct_red_generation = ("generation_color_clean",      lambda x: (x == 3).mean()),
    )
    .reset_index()
)

# ── NEW: Merge Industry Relevance into pc4 ------------------------------------
pc4 = pc4.merge(
    industry_df, 
    left_on="pc4", 
    right_on="postal code", 
    how="left"
)

# Fill areas that aren't on our specialized industry list with a base score of 0
pc4["industry_score"] = pc4["industry_score"].fillna(0)
pc4["industry"] = pc4["industry"].fillna("Standard/Mixed")
pc4["relevance"] = pc4["relevance"].fillna("Unrated")


# ── Step 3: Composite pc4 priority score (Updated Weights) --------------------
# fallback if sklearn unavailable:
def _mm(s): 
    denom = s.max() - s.min()
    return (s - s.min()) / (denom if denom != 0 else 1e-9)

# New scoring formula includes the heavy-industry signal. 
# Weights adjusted to accommodate the new 25% factor.
pc4["pc4_score"] = (
    _mm(pc4["mean_val_score"])      * 0.35 +  # Base area priority (35%)
    _mm(pc4["pct_red_consumption"]) * 0.15 +  # Congestion confirmation (20%)
    _mm(pc4["pc6_count"])           * 0.15 +  # Zone size/density (15%)
    _mm(pc4["industry_score"])      * 0.35    # High energy density industry (30%)
)

# Rank within each supply area (so every region surfaces its best zones)
pc4["rank_in_area"] = (
    pc4.groupby("supply_area_id")["pc4_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Global rank across all pc4 zones in the top-10 areas
pc4["global_rank"] = pc4["pc4_score"].rank(ascending=False, method="min").astype(int)


# ── Output: top 5 pc4 zones per area ------------------------------------------
top_pc4 = (
    pc4[pc4["rank_in_area"] <= 5]
    .sort_values(["global_rank"])
    [["global_rank", "rank_in_area", "pc4", "supply_area_name", 
      "regional_grid_operator", "industry", "relevance",  # Standardized to lowercase
      "pc6_count", "mean_val_score", "pct_red_consumption", 
      "pct_red_generation", "pc4_score"]]
    .reset_index(drop=True)
)

print(f"\nTOP 5 pc4 ZONES PER AREA  ({len(top_pc4)} zones across 10 areas)")
display(top_pc4)

# Save for next step (enterprise search)
top_pc4.to_csv("outputs/top_pc4_prospecting_zones.csv", index=False)
print("\n✓  outputs/top_pc4_prospecting_zones.csv")

TOP 10 TIER-1 SUPPLY AREAS


,supply_area_id,regional_grid_operator,valorization_score,valorization_rank,consumption_utilisation_ratio,feedin_utilisation_ratio,total_queue_mw,near_term_project_count
1,OS WEESP 10-9i,Liander,0.5688,1,10000000000.0000,0.2500,NaN,0
2,Medemblik Transformator 1,Liander,0.4702,2,1000000000.0000,0.3500,NaN,0
3,OS ULFT 20-3i,Liander,0.2679,3,0.2826,1.4783,105.4766,0
4,MZ,Enexis,0.2351,4,0.5570,1.3508,188.6460,0
5,VENR,Enexis,0.1870,5,0.4785,0.6173,395.4780,0
6,ISM,Enexis,0.1846,6,0.2978,0.9342,83.8010,0
7,VO,Enexis,0.1697,7,0.8682,1.2026,32.8190,0
8,ZLH,Enexis,0.1615,8,0.7744,0.8848,142.6160,0
9,CM56/78,Stedin,0.1610,9,1.0146,2.0000,NaN,0
10,CM08,Stedin,0.1586,10,NaN,1.6200,8.4120,0



TOP 5 pc4 ZONES PER AREA  (42 zones across 10 areas)


,global_rank,rank_in_area,pc4,supply_area_name,regional_grid_operator,industry,relevance,pc6_count,mean_val_score,pct_red_consumption,pct_red_generation,pc4_score
0,1,1,1771,Medemblik Transformator 1,Liander,Hyperscale Data Centers (Google/MSFT) & Agri-l...,3 - High (Modern EII),4,0.4702,1.0000,0.0000,0.6973
1,2,2,1775,Medemblik Transformator 1,Liander,Hyperscale Data Centers (Google/MSFT) & Agri-l...,3 - High (Modern EII),3,0.4702,1.0000,0.0000,0.6968
2,3,1,1398,OS WEESP 10-9i,Liander,E-commerce Fulfillment & Office Parks,10 - Very Low,127,0.5688,1.0000,0.0000,0.5944
3,4,1,7833,VO Veenoord,Enexis,Chemical & Polymers (Getec Industrial Park),1 - Extremely High,165,0.1697,1.0000,1.0000,0.5868
4,5,2,7887,VO Veenoord,Enexis,Chemical & Polymers (Getec Industrial Park),1 - Extremely High,133,0.1697,1.0000,1.0000,0.5717
5,6,2,1384,OS WEESP 10-9i,Liander,E-commerce Fulfillment & Office Parks,10 - Very Low,54,0.5688,1.0000,0.0000,0.5600
6,7,3,7844,VO Veenoord,Enexis,Chemical & Polymers (Getec Industrial Park),1 - Extremely High,73,0.1697,1.0000,1.0000,0.5434
7,8,3,1382,OS WEESP 10-9i,Liander,E-commerce Fulfillment & Office Parks,10 - Very Low,1,0.5688,1.0000,0.0000,0.5350
8,9,1,6021,MZ Maarheeze,Enexis,Heavy Manufacturing & Metal Processing,2 - High (Concentrated),275,0.2351,0.0000,1.0000,0.5095
9,10,2,6026,MZ Maarheeze,Enexis,Metallurgy (Nyrstar Zinc Smelter),2 - High (Concentrated),166,0.2351,0.0000,1.0000,0.4581



✓  outputs/top_pc4_prospecting_zones.csv


In [47]:
print (top_pc4['pc4'].tolist()[:10])


['1771', '1775', '1398', '7833', '7887', '1384', '7844', '1382', '6021', '6026']


---
## 16 · Sanity-Check the pc4 Results

Before moving to enterprise search, two quick checks:

1. **Which supply area owns each pc4?** — confirms the 1xxx cluster is genuinely one area and not an artifact of aggregation across areas
2. **Coverage check** — how many of the 10 areas contributed zones, and which ones are underrepresented (< 5 zones = small area, fine; 0 zones = join issue)


In [48]:
# ── Check 1: Area breakdown of the top 42 pc4 zones ---------------------------
area_coverage = (
    top_pc4.groupby(["supply_area_name", "regional_grid_operator"])
    .agg(
        zones_in_top    = ("pc4",        "count"),
        pc4_list        = ("pc4",        lambda x: sorted(x.tolist())),
        best_pc4_score  = ("pc4_score",  "max"),
        total_pc6_count = ("pc6_count",  "sum"),
    )
    .reset_index()
    .sort_values("best_pc4_score", ascending=False)
    .reset_index(drop=True)
)
area_coverage.index += 1

print("AREA COVERAGE — zones contributed per supply area")
display(area_coverage)

# ── Check 2: pc4 geography context --------------------------------------------
# The pc4 prefix maps to a Dutch municipality/district.
# Quick lookup table for the top 10 pc4 codes based on Dutch postcode ranges.
PC4_REGION_HINT = {
    # Amsterdam & Surrounds (North Holland)
    "10": "Amsterdam",
    "11": "Amsterdam regional (Amstelveen / Diemen / Purmerend)",
    "12": "Gooi area (Hilversum / Huizen)",
    "13": "Almere", 
    "14": "Haarlemmermeer (Schiphol / Amstelveen rural)",
    "17": "Noord-Holland Noord (Schagen / Den Helder rural)",
    "18": "Alkmaar area",

    # Haarlem, Leiden, & South-Holland Coast
    "20": "Haarlem",
    "21": "Heemstede / Hillegom / Bloembollenstreek",
    "22": "Katwijk / Noordwijk / Wassenaar",
    "23": "Leiden area",                                    # Added
    "24": "Alphen aan den Rijn / Gouda North",             # Added
    "25": "The Hague (Den Haag)",
    "26": "Delft / Rijswijk",
    "27": "Zoetermeer",
    "28": "Gouda / Waddinxveen",                           # Added

    # Rotterdam & Rijnmond
    "30": "Rotterdam",
    "31": "Schiedam / Vlaardingen / Spijkenisse",
    "32": "Voorne-Putten / Goeree-Overflakkee",
    "33": "Dordrecht / Zwijndrecht",

    # North Brabant & Limburg
    "50": "Tilburg area",
    "51": "Dongen / Waalwijk / Kaatsheuvel",
    "55": "Veldhoven / Kempen region (near Eindhoven)",
    "56": "Eindhoven",
    "57": "Helmond area",                                  # Added
    "58": "Venray / Boxmeer (North-Limburg / East-Brabant border)",
    "60": "Weert / Roermond (Central Limburg)",            # Added

    # Overijssel, Drente, Flevoland, & Gelderland
    "70": "Doetinchem / Achterhoek region",                # Added
    "71": "Winterswijk / East Achterhoek",                 # Added
    "77": "Dedemsvaart / Ommen / Hardenberg",              # Added
    "78": "Emmen / Southeast Drenthe",                     # Added
    "80": "Zwolle area",                                   # Added
    "82": "Lelystad / Dronten",                            # Added
}

top5_pc4 = top_pc4.head(100)["pc4"].tolist()

print("\nGEOGRAPHY CONTEXT — top 5 pc4 zones")
print(f"{'pc4':<8} {'Likely region'}")
print("-" * 45)
for pc4_code in top5_pc4:
    prefix2 = pc4_code[:2]
    hint = PC4_REGION_HINT.get(prefix2, "— add to lookup table")
    print(f"  {pc4_code:<6}  {hint}")

print()
print("Next step → use these pc4 codes to query KvK / OpenStreetMap / Google Places")
print("for industrial enterprises (SBI codes 35.1, 20.x, 24.x, 27.x for energy-intensive industries)")

AREA COVERAGE — zones contributed per supply area


,supply_area_name,regional_grid_operator,zones_in_top,pc4_list,best_pc4_score,total_pc6_count
1,Medemblik Transformator 1,Liander,2,"[1771, 1775]",0.6973,7
2,OS WEESP 10-9i,Liander,3,"[1382, 1384, 1398]",0.5944,182
3,VO Veenoord,Enexis,5,"[7761, 7765, 7833, 7844, 7887]",0.5868,567
4,MZ Maarheeze,Enexis,5,"[5591, 5595, 5712, 6021, 6026]",0.5095,1018
5,VENR Venray,Enexis,5,"[5801, 5802, 5825, 5854, 5855]",0.4538,821
6,ISM IJsselmuiden,Enexis,5,"[8267, 8269, 8271, 8274, 8277]",0.3932,321
7,OS ULFT 20-3i,Liander,5,"[7051, 7077, 7078, 7081, 7122]",0.3916,221
8,Stompwijk en Leidschendam-Zuid,Stedin,3,"[2266, 2381, 2492]",0.2824,131
9,ZLH Zwolle Hessenweg,Enexis,5,"[7721, 7722, 8024, 8028, 8035]",0.2755,490
10,Waddinxveen,Stedin,4,"[2741, 2742, 2743, 2803]",0.2200,720



GEOGRAPHY CONTEXT — top 5 pc4 zones
pc4      Likely region
---------------------------------------------
  1771    Noord-Holland Noord (Schagen / Den Helder rural)
  1775    Noord-Holland Noord (Schagen / Den Helder rural)
  1398    Almere
  7833    Emmen / Southeast Drenthe
  7887    Emmen / Southeast Drenthe
  1384    Almere
  7844    Emmen / Southeast Drenthe
  1382    Almere
  6021    Weert / Roermond (Central Limburg)
  6026    Weert / Roermond (Central Limburg)
  5801    Venray / Boxmeer (North-Limburg / East-Brabant border)
  5712    Helmond area
  7761    Dedemsvaart / Ommen / Hardenberg
  5591    Veldhoven / Kempen region (near Eindhoven)
  8271    Lelystad / Dronten
  7081    Doetinchem / Achterhoek region
  5802    Venray / Boxmeer (North-Limburg / East-Brabant border)
  5854    Venray / Boxmeer (North-Limburg / East-Brabant border)
  5825    Venray / Boxmeer (North-Limburg / East-Brabant border)
  7078    Doetinchem / Achterhoek region
  7765    Dedemsvaart / Ommen / Harde